# Dynamic programming — three checkpoints
DP = recursion + memory. We reuse overlapping subproblems instead of recomputing.
Checkpoints: (1) Fibonacci, (2) edit distance, (3) trellis.
See slides: Part 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(0)   # fixed seed -> reproducible for everyone

np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)


## Checkpoint 1 — Fibonacci: caching kills the exponential
Naive recursion recomputes the same subproblems exponentially often; a cache
makes it linear. See slide: *Example 1 — Fibonacci*.

In [ ]:
def fib_naive(n):
    if n < 2:
        return n
    return fib_naive(n-1) + fib_naive(n-2)


In [ ]:
def fib_memo(n, cache=None):
    if cache is None:
        cache = {}
    if n < 2:
        return n
    if n not in cache:
        cache[n] = fib_memo(n-1, cache) + fib_memo(n-2, cache)   # cache miss: compute once, store
    return cache[n]


In [ ]:
def fib_bu(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b       # shift the window: no recursion, O(1) space
    return a


In [ ]:
for k in range(15):
    assert fib_memo(k) == fib_naive(k) == fib_bu(k)
assert fib_memo(50) == 12586269025
print("correctness ✓")

import time
t0 = time.perf_counter(); fib_naive(32); t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); fib_memo(32);  t_memo  = time.perf_counter() - t0
t0 = time.perf_counter(); fib_bu(32); t_bu = time.perf_counter() - t0
print(f"naive fib(32): {t_naive:.4f}s   memo fib(32): {t_memo:.6f}s   bu fib(32): {t_bu}")
assert t_memo < t_naive
print("speedup ✓  (this is overlapping subproblems, paid for once)")


## Checkpoint 2 — Edit distance: 2D table, min-over-choices, traceback
D[i][j] = edit distance between prefixes a[:i] and b[:j]. Three new ideas vs
Fibonacci: 2D state, a genuine choice (insert/delete/substitute), and recovering
the solution via backpointers. This DP *is* sequence alignment. See slide:
*Example 2 — Edit distance*.

In [ ]:
def edit_distance(a, b):
    m, n = len(a), len(b)
    D = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): D[i][0] = i     # delete i chars
    for j in range(n+1): D[0][j] = j     # insert j chars
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                D[i][j] = D[i-1][j-1]                  # free match
            else:
                D[i][j] = 1 + min(D[i-1][j],      # delete a[i-1]
                                  D[i][j-1],      # insert b[j-1]
                                  D[i-1][j-1])    # substitute
    return D[m][n]


In [ ]:
assert edit_distance("kitten", "sitting") == 3
assert edit_distance("", "abc") == 3
assert edit_distance("abc", "abc") == 0
assert edit_distance("flaw", "lawn") == 2
print("passed ✓")


In [ ]:
def edit_table(a, b):
    m, n = len(a), len(b)
    D = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): D[i][0] = i
    for j in range(n+1): D[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            D[i][j] = min(D[i-1][j]+1, D[i][j-1]+1, D[i-1][j-1]+cost)
    return np.array(D)

b_ = "sitting"
print("     " + "  ".join(b_))
print(edit_table("kitten", b_))


### Recover the edits [ADVANCED]
Store a backpointer per cell (which of match/delete/insert/substitute you took),
then walk from D[m][n] back to D[0][0] to print the operations. 

In [ ]:
def edit_ops(a, b):
    m, n = len(a), len(b)
    D = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): D[i][0] = i
    for j in range(n+1): D[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            D[i][j] = min(D[i-1][j]+1, D[i][j-1]+1, D[i-1][j-1]+cost)
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i][j] == D[i-1][j-1] + (0 if a[i-1]==b[j-1] else 1):
            ops.append("match" if a[i-1]==b[j-1] else f"sub {a[i-1]}->{b[j-1]}")
            i, j = i-1, j-1
        elif i > 0 and D[i][j] == D[i-1][j] + 1:
            ops.append(f"del {a[i-1]}"); i -= 1
        else:
            ops.append(f"ins {b[j-1]}"); j -= 1
    return list(reversed(ops))


In [ ]:
ops = edit_ops("kitten", "sitting")
assert sum(not o.startswith("match") for o in ops) == 3
print(ops); print("passed ✓")


## Checkpoint 3 — Min-cost path through a trellis
T time steps, K states each. node_costs[t, j] = cost of state j at time t;
trans_costs[i, j] = cost of moving i -> j. Find the cheapest path picking one
state per step, and recover it with backpointers.

Keep this in mind: with cost = -log(probability), THIS IS VITERBI.
See slide: *Example 3 — Trellis*.

In [ ]:
def min_cost_path(node_costs, trans_costs):
    """node_costs: (T, K); trans_costs: (K, K) with trans_costs[i, j] = cost i->j.
    Returns (best_cost, best_path) where best_path is a list of T state indices."""
    T, K = node_costs.shape
    V  = np.full((T, K), np.inf)   # V[t, j] = best cost of a path ending at (t, j)
    bp = np.zeros((T, K), dtype=int)
    V[0] = node_costs[0]           # base case
    for t in range(1, T):
        for j in range(K):
            candidates = V[t-1] + trans_costs[:, j]     # cost to arrive at j via each predecessor i
            bp[t, j] = int(np.argmin(candidates))
            V[t, j]  = node_costs[t, j] + candidates[bp[t, j]]
    # backtrack from the cheapest final state
    best_last = int(np.argmin(V[-1]))
    path = [best_last]
    for t in range(T-1, 0, -1):
        path.append(int(bp[t, path[-1]]))
    return float(V[-1, best_last]), list(reversed(path))


In [ ]:
# Test 1: transitions free -> pick the cheapest node each step
nc = np.array([[0., 5.], [1., 1.], [0., 4.]])
tc = np.zeros((2, 2))
cost, path = min_cost_path(nc, tc)
assert np.isclose(cost, 1.0), cost
assert path == [0, 0, 0], path

# Test 2: nodes free, switching expensive -> path must stay put
nc = np.zeros((3, 2))
tc = np.array([[0., 10.], [10., 0.]])
cost, path = min_cost_path(nc, tc)
assert np.isclose(cost, 0.0) and len(set(path)) == 1, (cost, path)
print("passed ✓")


You now have every ingredient for HMM inference:
a table keyed by (time, state), a recurrence that reuses the previous column,
and backpointers. Next notebook: give these a probabilistic meaning.